In [1]:
# Read dataset

import gzip
import json

with gzip.open('../../data/hooktheory/Hooktheory.json.gz', 'r') as f:
    dataset = json.load(f)

print(len(dataset))

26175


In [2]:
dataset_split_all = {}
for split in ["train", "valid", "test"]:
    dataset_split = {
        k: v
        for k, v in dataset.items()
        if v["split"] == split.upper()
        and "MELODY" in v["tags"]
        and "HARMONY" in v["tags"]
        and "TEMPO_CHANGES" not in v["tags"]
    }
    dataset_split = list(dataset_split.values())
    dataset_split_all[split] = dataset_split

train_set = dataset_split_all['train']
print(len(train_set))

19052


In [4]:
# Inspect one

example = train_set[14715]
print(type(example))
print(json.dumps(example, indent=2))

<class 'dict'>
{
  "tags": [
    "KEY_CHANGES",
    "NO_SWING",
    "AUDIO_AVAILABLE",
    "USER_ALIGNMENT",
    "REFINED_ALIGNMENT",
    "MELODY",
    "HARMONY"
  ],
  "split": "TRAIN",
  "hooktheory": {
    "id": "ZbgOYRDMonY",
    "artist": "paul-mccartney",
    "song": "when-the-wind-is-blowing",
    "annotators": [
      "Dr_Jackhammer",
      "atp90210",
      "Vaz123"
    ],
    "urls": {
      "artist": "https://www.hooktheory.com/theorytab/artists/p/paul-mccartney",
      "song": "https://www.hooktheory.com/theorytab/view/paul-mccartney/when-the-wind-is-blowing",
      "clip": "https://hookpad.hooktheory.com/?idOfSong=ZbgOYRDMonY"
    }
  },
  "youtube": {
    "id": "5LV70mRhkrw",
    "url": "https://www.youtube.com/watch?v=5LV70mRhkrw",
    "duration": 231.65333333333334
  },
  "alignment": {
    "swing": "STRAIGHT",
    "user": {
      "beats": [
        0,
        60
      ],
      "times": [
        31.765930734961408,
        79.1147555598466
      ]
    },
    "refined":

In [ ]:
# Parse alignment

import numpy as np
from scipy.interpolate import interp1d

beat_to_time_fn = interp1d( # map a given beat to the absolute time
    example['alignment']['refined']['beats'],   # how many beats in the song
    example['alignment']['refined']['times'],   # at what absolute time [s] the beats occur
    kind='linear',
    fill_value='extrapolate') 
start_time = beat_to_time_fn(0)
num_beats = example['annotations']['num_beats'] 
end_time = beat_to_time_fn(num_beats)
print(start_time, end_time)

for i in range(32):
    print(beat_to_time_fn(i))

In [ ]:
# Interpret annotation as MIDI

CHORD_OCTAVE = 4
MELODY_OCTAVE = 4

import pretty_midi

midi = pretty_midi.PrettyMIDI()

# Create click track
# click = pretty_midi.Instrument(program=0, is_drum=True)
# midi.instruments.append(click)
# beats_per_bar = example['annotations']['meters'][0]['beats_per_bar']
# for b in range(num_beats + 1):
#     downbeat = b % beats_per_bar == 0
#     click.notes.append(pretty_midi.Note(
#         100 if downbeat else 75, 
#         37 if downbeat else 31,
#         beat_to_time_fn(b),
#         beat_to_time_fn(b + 1)))

# Create harmony track
harmony = pretty_midi.Instrument(program=0)
midi.instruments.append(harmony)
for c in example['annotations']['harmony']:
    root_position_pitches = [c['root_pitch_class']]
    for interval in c['root_position_intervals']:
        root_position_pitches.append(root_position_pitches[-1] + interval) # Put all notes of the chord into the list
    for p in root_position_pitches: # Write the notes of the chord to the midi track
        harmony.notes.append(pretty_midi.Note(  # Note(velocity, pitch, onset, offset)
            67,
            p + CHORD_OCTAVE * 12,
            beat_to_time_fn(c['onset']),
            beat_to_time_fn(c['offset'])
        ))

# Create melody track
melody = pretty_midi.Instrument(program=0)
midi.instruments.append(melody)
for n in example['annotations']['melody']:
    melody.notes.append(pretty_midi.Note(     # Note(velocity, pitch, onset, offset)
        100,
        n['pitch_class'] + (MELODY_OCTAVE + n['octave']) * 12,
        beat_to_time_fn(n['onset']),
        beat_to_time_fn(n['offset'])
    ))

midi.write(f"midis/annotations_{example['hooktheory']['song']}_{example['hooktheory']['id']}.midi")

In [ ]:
# Retrieve and decode audio (this one is CC-licensed)
# Thanks JohnJRenns https://coolandnewwebcomic.bandcamp.com/track/plague !

import librosa
from IPython.display import display, Audio

!youtube-dl --format bestaudio/best --output 'audio' {example['youtube']['url']}
audio, sr = librosa.load('audio', sr=None, mono=True, offset=start_time, duration=end_time - start_time)

In [ ]:
# Synthesize aligned preview

annotations_audio = midi.fluidsynth(fs=sr)
annotations_audio = annotations_audio[round(start_time * sr):]
annotations_audio = annotations_audio[:audio.shape[0]]
if annotations_audio.shape[0] < audio.shape[0]:
    annotations_audio = np.pad(annotations_audio, [(0, audio.shape[0] - annotations_audio.shape[0])])
display(Audio([audio, annotations_audio], rate=sr))